# Taller 1 — Notebook 02: Estrategia de análisis

**MINE-4101 · Ciencia de Datos Aplicada**

Cubre el **punto 2 (15 %)** del taller: describir la estrategia de análisis y las técnicas
empleadas, con su justificación.

**Requisito:** ejecutar antes [`01_entendimiento_datos.ipynb`](01_entendimiento_datos.ipynb),
que genera `data/processed/contratos_limpio.parquet`.

## La estrategia, en dos párrafos

El requerimiento del negocio es decidir **a qué contratos hacer seguimiento estrecho desde el
momento de la firma**. Eso impone dos restricciones que gobiernan toda la estrategia. La
primera es que el objetivo debe ser una desviación *observable y con prevalencia suficientemente
baja para poder focalizar*: el notebook 01 mostró que las tres definiciones literales del
enunciado no cumplen eso —el compuesto se activa en el 83,77 % de los contratos— y por eso el
objetivo se redefinió como `desviacion` = adición de plazo **o** subejecución parcial real
(13,55 %), esta última evaluada solo donde el plazo ya venció. La segunda es que los criterios solo pueden usar variables **conocidas a la firma**:
`valor_pagado`, `dias_adicionados`, `estado_contrato`, `liquidacion` y `fecha_de_fin` son
posteriores al hecho, y usarlas produciría un criterio que "anticipa" con información que el
supervisor no tendrá cuando deba decidir. La estrategia procede entonces en tres escalones:
estadística descriptiva para establecer tasas base por segmento, contraste formal de hipótesis
bivariadas para determinar qué atributos discriminan, y un cruce de los atributos que superan el
umbral, que separa el efecto propio de cada uno del de la composición y se convierte en la regla
de focalización.

**La prueba de cada hipótesis se elige por el tipo de las dos variables que cruza**, siguiendo
el árbol de decisión del curso, y no por conveniencia. Como los datos son abrumadoramente
categóricos —solo 5 de 36 variables son numéricas— el instrumento dominante resulta ser el
**chi-cuadrado de independencia** sobre tablas de contingencia: nueve de los doce contrastes.
Los otros tres son un **Z test** de dos medias para comparar el valor de los contratos que se
desvían frente a los que no, una **correlación** entre el nivel de valor y la tasa de
desviación, y un **ANOVA de Welch** para la magnitud de los días adicionados entre sectores.
Cada prueba declara y verifica su supuesto, y en las tres donde la asimetría de los datos podría
preocupar se añade la verificación no paramétrica equivalente para comprobar que la conclusión
no depende de esa asimetría. El criterio de decisión **no es el valor p** sino el tamaño de
efecto contrastado contra un umbral fijado antes de mirar los datos, porque con n ≈ 196.000
diferencias sin importancia práctica alcanzan significancia. El cierre **no agrega técnicas
nuevas**: repite el chi-cuadrado dentro de cada rango de valor para separar el efecto propio
de cada atributo del de la composición, cruza el valor y la modalidad en una **tabla de tasas
con su intervalo de confianza** y mide cuántas desviaciones detecta la oficina según cuántos
contratos revise. Todo el análisis usa las herramientas vistas en clase; no se usan modelos
predictivos.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import config
from src.carga import cargar_limpio
from src.inferencia import agrupar_raras, v_cramer
from src.viz import aplicar_estilo, guardar

aplicar_estilo()
config.asegurar_directorios()
pd.set_option("display.width", 180)

df = cargar_limpio()
print(f"{df.shape[0]:,} filas x {df.shape[1]} columnas")

  dataset limpio: 196,391 filas x 70 columnas


196,391 filas x 70 columnas


---
## 1. Definición operativa del objetivo

Recapitulación de lo establecido en el notebook 01. Las cifras se recalculan aquí para que este
notebook sea verificable por sí solo.

In [2]:
n = len(df)
senales = pd.DataFrame([
    {"señal": "tiene_adicion", "definicion": "dias_adicionados > 0",
     "n": int(df["tiene_adicion"].sum()), "observable_en": "todo el dataset"},
    {"señal": "subejecucion_parcial", "definicion": "0 < pagado/valor < 0,90 y plazo vencido",
     "n": int(df["subejecucion_parcial"].sum()),
     "observable_en": "pago reportado y plazo vencido"},
    {"señal": "sin_liquidar", "definicion": "liquidación = No",
     "n": int(df["sin_liquidar"].sum()), "observable_en": "descriptiva, no objetivo"},
    {"señal": "pago_cero", "definicion": "pagado = 0",
     "n": int(df["pago_cero"].sum()), "observable_en": "zona ciega"},
    {"señal": "desviacion (OBJETIVO)", "definicion": "adición O subejecución parcial",
     "n": int(df["desviacion"].sum()), "observable_en": "todo el dataset"},
    {"señal": "desviacion_severa", "definicion": "n_senales >= 2",
     "n": int(df["desviacion_severa"].sum()), "observable_en": "alternativa exigente"},
])
senales["prevalencia_%"] = (100 * senales["n"] / n).round(2)
display(senales)

,señal,definicion,n,observable_en,prevalencia_%
0,tiene_adicion,dias_adicionados > 0,18671,todo el dataset,9.51
1,subejecucion_parcial,"0 < pagado/valor < 0,90 y plazo vencido",9355,pago reportado y plazo vencido,4.76
2,sin_liquidar,liquidación = No,118418,"descriptiva, no objetivo",60.30
3,pago_cero,pagado = 0,107452,zona ciega,54.71
4,desviacion (OBJETIVO),adición O subejecución parcial,26603,todo el dataset,13.55
5,desviacion_severa,n_senales >= 2,15400,alternativa exigente,7.84


### Por qué `desviacion` y no las otras

- **`sin_liquidar` (60,30 %) queda fuera del objetivo.** Un indicador que señala 6 de cada 10
  contratos describe la norma administrativa, no una anomalía. Se conserva como variable
  descriptiva y dentro de `n_senales`.
- **`pago_cero` (54,71 %) no es evidencia de incumplimiento.** Mezcla contratos genuinamente no
  ejecutados con contratos cuyo pago nunca se reportó, y los datos no permiten separarlos. En vez
  de descartarlo, se convierte en **pregunta propia** (H8): ¿el fallo de reporte se concentra en
  algún segmento? Para una oficina de control interno, un contrato sin registro de pago es en sí
  mismo un problema de supervisión.
- **`desviacion` (13,55 %) es el objetivo principal** y `desviacion_severa` (7,84 %) queda como
  alternativa si la oficina necesita focalizar aún más.

---
## 2. Muestras de análisis

No todas las hipótesis se contrastan sobre las mismas filas, porque no todas las señales son
observables en el mismo subconjunto. Declararlo explícitamente evita comparar tasas calculadas
sobre universos distintos.

In [3]:
M_TOTAL = df
M_PAGO = df[(df["pct_ejecucion_financiera"] > 0) & df["desenlace_observable"]]

muestras = pd.DataFrame([
    {"muestra": "M_TOTAL", "n": len(M_TOTAL),
     "criterio": "todas las filas",
     "se_usa_para": "tiene_adicion, desviacion, pago_cero"},
    {"muestra": "M_PAGO", "n": len(M_PAGO),
     "criterio": "pct_ejecucion_financiera > 0 Y plazo vencido",
     "se_usa_para": "subejecucion_parcial (la señal financiera)"},
])
muestras["% del total"] = (100 * muestras["n"] / len(df)).round(2)
display(muestras)

print(f"zona ciega excluida de M_PAGO: {int(df['pago_cero'].sum()):,} contratos")
print("\nRegla: la senal financiera NUNCA se evalua sobre la zona ciega, porque ahi")
print("el cero no distingue entre no ejecutar y no reportar.")
print("")
no_obs = ~df["desenlace_observable"]
sin_fin = df["fecha_de_fin_del_contrato"].isna()
print(f"no observables, excluidos de la senal financiera: {int(no_obs.sum()):,} contratos")
print(f"  {int((no_obs & ~sin_fin).sum()):,} con plazo vigente y "
      f"{int((no_obs & sin_fin).sum()):,} con fecha de fin anulada por imposible")
print("Un contrato en curso que lleva pagado el 40 % no esta subejecutado:")
print("todavia no ha terminado. Su desenlace no es observable.")

,muestra,n,criterio,se_usa_para,% del total
0,M_TOTAL,196391,todas las filas,"tiene_adicion, desviacion, pago_cero",100.00
1,M_PAGO,88363,pct_ejecucion_financiera > 0 Y plazo vencido,subejecucion_parcial (la señal financiera),44.99


zona ciega excluida de M_PAGO: 107,452 contratos

Regla: la senal financiera NUNCA se evalua sobre la zona ciega, porque ahi
el cero no distingue entre no ejecutar y no reportar.

no observables, excluidos de la senal financiera: 948 contratos
  928 con plazo vigente y 20 con fecha de fin anulada por imposible
Un contrato en curso que lleva pagado el 40 % no esta subejecutado:
todavia no ha terminado. Su desenlace no es observable.


---
## 3. Hipótesis a contrastar

Primero, las ocho preguntas en palabras simples:

| # | La pregunta | Herramienta |
|---|---|---|
| H1 | ¿Alargar el plazo depende de cómo se contrató (licitación, mínima cuantía, directa…)? | Chi-cuadrado |
| H2 | Si se ordenan los contratos del más barato al más caro, ¿la desviación sube con el precio? | Correlación |
| H2b | ¿Los contratos que se desvían valen más que los que no? | Z test |
| H3 | ¿Pagar solo una parte pasa más en gasto de inversión o en gasto de funcionamiento? | Chi-cuadrado |
| H4 | ¿Alargar el plazo depende de si la entidad es nacional o territorial? ¿Y del sector? | Chi-cuadrado (dos veces) |
| H5 | ¿Se alargan más los contratos que dicen «prorrogable»? ¿Y los que permiten pago adelantado? | Chi-cuadrado (dos veces) |
| H6 | Entre los contratos que sí se alargaron, ¿el número de días añadidos cambia según el sector? | ANOVA |
| H7 | Control: ¿el género del representante legal tiene que ver con la desviación? | Chi-cuadrado |
| H8 | ¿Dejar de registrar pagos se concentra en ciertos tipos de entidad o sectores? | Chi-cuadrado (dos veces) |

Son ocho hipótesis pero doce pruebas, porque H4, H5 y H8 hacen dos preguntas cada una y H2 tiene
dos versiones.

**¿Por qué hacer preguntas que se espera que den «no»?** H7 y la parte del pago adelantado de H5
son como el placebo de un experimento médico. Si el género saliera «importante», querría decir
que el método está detectando cosas que no existen. Además, el enunciado pide reportar también lo
que no resultó.

Abajo, la misma lista en su forma técnica, con la muestra, la prueba, el tamaño de efecto y la
verificación de robustez de cada una.

| # | Hipótesis (H₁) | Objetivo | Muestra | Tipo de cruce | Prueba | Tamaño de efecto | Robustez |
|---|---|---|---|---|---|---|---|
| **H1** | La tasa de adición de plazo difiere entre modalidades de contratación | `tiene_adicion` | M_TOTAL | categórica × categórica | **χ²** | V de Cramér | — |
| **H2** | La tasa de desviación crece con el nivel de valor del contrato | `desviacion` | M_TOTAL, por deciles | numérica × numérica | **Correlación** | r con IC 95 % | Spearman |
| **H2b** | Los contratos que se desvían tienen mayor valor | `desviacion` | M_TOTAL | numérica × categórica (2 grupos, n grande) | **Z test** | diferencia de medias con IC 95 %, leída como factor | Mann-Whitney |
| **H3** | La subejecución parcial difiere entre destinos del gasto | `subejecucion_parcial` | M_PAGO | categórica × categórica | **χ²** | diferencia en pp con IC 95 % | — |
| **H4** | La adición difiere por orden de la entidad y por sector | `tiene_adicion` | M_TOTAL | categórica × categórica | **χ²** | V de Cramér | — |
| **H5** | La prorrogabilidad y el pago adelantado se asocian a la adición | `tiene_adicion` | M_TOTAL | categórica × categórica (2×2) | **χ²** | V de Cramér y diferencia en pp con IC 95 % | — |
| **H6** | La *magnitud* de los días adicionados difiere entre sectores | `dias_adicionados` | M_TOTAL, solo con adición | numérica × categórica (>2 grupos) | **ANOVA de Welch** | η² | Kruskal-Wallis |
| **H7** | **Control:** el género del representante legal se asocia a la desviación | `desviacion` | M_TOTAL | categórica × categórica | **χ²** | V de Cramér | — |
| **H8** | El fallo de reporte de pagos se concentra por orden y sector | `pago_cero` | M_TOTAL | categórica × categórica | **χ²** | V de Cramér | — |

Y un cierre que no agrega pruebas nuevas:

| Paso | Herramienta | Para qué |
|---|---|---|
| **Control por composición** (notebook 03, §12) | χ² dentro de cada rango de valor, y tasa observada frente a la esperada por la mezcla de valor y modalidad | Separar el efecto propio del destino del gasto, el sector y el año del efecto de la composición |
| **Tabla de riesgo** (notebook 04) | Tasa de desviación de cada combinación valor × modalidad, con IC 95 % | Ordenar los contratos y medir cuántas desviaciones detecta la oficina según cuántos revise |

> Todo el análisis usa las herramientas vistas en clase: las cinco pruebas del árbol de decisión,
> intervalos de confianza y estadística descriptiva. No se usan modelos predictivos.

**Cómo se lee cada prueba.** Cada hipótesis se plantea como un par: H₀ dice que no hay relación
y H₁ que sí la hay. Funciona como un juicio con presunción de inocencia: se parte de H₀ y solo se
abandona si la evidencia es fuerte. El valor p responde a esta pregunta: si en realidad no hubiera ninguna relación, ¿qué tan raro
sería ver, solo por casualidad, una diferencia como la observada o mayor? Si es menor que 0,05
(menos de 1 en 20), el resultado es estadísticamente significativo: es difícil explicarlo como
casualidad, y se rechaza H₀.

**Verificaciones de robustez.** Se reportan aparte de las doce pruebas primarias: no son
contrastes independientes, sino la misma pregunta hecha con otra prueba.

**El criterio de decisión no es el valor p.** Significativo no quiere decir importante: con
n ≈ 196.000, una diferencia de 0,4 puntos porcentuales ya es difícil de atribuir a la casualidad,
aunque no sirva para decidir nada. Una hipótesis se considera **útil para focalizar** solo si
(a) es significativa (p < 0,05), (b) su tamaño de efecto —cuánto cambia de verdad la tasa de un
grupo a otro— supera el umbral fijado a priori (sección 4), y
(c) el segmento tiene volumen suficiente para que supervisarlo sea operativamente viable.

In [4]:
# Verificacion previa de supuestos: ¿que variables tienen categorias demasiado
# raras para un chi-cuadrado? Se resuelve agrupando en "Otros" antes de probar.
print("Categorias con menos del 1 % de los registros (se agruparan):\n")
for var in ["modalidad_de_contratacion", "sector", "orden", "destino_gasto",
            "el_contrato_puede_ser_prorrogado", "habilita_pago_adelantado",
            "genero_representante_legal"]:
    s = df[var]
    share = s.value_counts(normalize=True)
    raras = share[share < 0.01]
    total = int((s.isin(raras.index)).sum())
    estado = f"{len(raras)} categorias, {total:,} registros" if len(raras) else "ninguna"
    print(f"  {var:34} {estado}")

Categorias con menos del 1 % de los registros (se agruparan):

  modalidad_de_contratacion          1 categorias, 236 registros
  sector                             12 categorias, 10,124 registros
  orden                              ninguna
  destino_gasto                      ninguna
  el_contrato_puede_ser_prorrogado   ninguna
  habilita_pago_adelantado           1 categorias, 335 registros
  genero_representante_legal         ninguna


In [5]:
# Tasas base por segmento: el punto de partida descriptivo sobre el que se
# miden todas las diferencias posteriores.
print(f"TASA BASE de desviacion: {100*df['desviacion'].mean():.2f} %\n")

for var in ["modalidad_de_contratacion", "destino_gasto", "orden", "tipo_de_contrato"]:
    g = (df.groupby(agrupar_raras(df[var]), observed=True)["desviacion"]
         .agg(n="size", tasa="mean").sort_values("tasa", ascending=False))
    g["tasa_%"] = (100 * g["tasa"]).round(2)
    print(f"--- {var} ---")
    print(g[["n", "tasa_%"]].to_string())
    print()

TASA BASE de desviacion: 13.55 %

--- modalidad_de_contratacion ---
                                                  n  tasa_%
modalidad_de_contratacion                                  
Licitación pública                             2330   34.25
Otros                                           236   30.93
Contratación régimen especial (con ofertas)    6154   21.25
Selección abreviada subasta inversa           32907   20.55
Selección abreviada de menor cuantía           8822   19.51
Contratación directa (con ofertas)             5001   12.52
Contratación directa                           3938   11.63
Mínima cuantía                               117753   10.91
Contratación régimen especial                 19250   10.46

--- destino_gasto ---
                     n  tasa_%
destino_gasto                 
Inversión        78383   15.24
Funcionamiento  117097   12.44

--- orden ---
                          n  tasa_%
orden                              
Nacional              96680   13.74
Te

---
## 4. Correspondencia con el árbol de decisión

El árbol del curso asigna la prueba según el tipo de las dos variables que se cruzan. Esta es la
correspondencia completa entre sus ramas y los contrastes del taller, que es la forma de
verificar que ninguna prueba se eligió por fuera del inventario.

In [6]:
# El T Test solo aplicaria si algun grupo de H2b fuera pequeno.
n_min = int(min(df["desviacion"].sum(), (~df["desviacion"]).sum()))

arbol = pd.DataFrame([
    {"rama del arbol": "numerica vs numerica",
     "prueba": "Correlation Test", "donde se usa": "H2 (decil de valor vs tasa)"},
    {"rama del arbol": "numerica vs categorica, <=2 grupos, muestras grandes",
     "prueba": "Z Test", "donde se usa": "H2b (log10 del valor, con y sin desviacion)"},
    {"rama del arbol": "numerica vs categorica, <=2 grupos, muestras pequenas",
     "prueba": "T Test", "donde se usa": f"no aplica: ningun grupo baja de {n_min:,} obs."},
    {"rama del arbol": "numerica vs categorica, >2 grupos",
     "prueba": "ANOVA", "donde se usa": "H6 (dias adicionados por sector)"},
    {"rama del arbol": "categorica vs categorica",
     "prueba": "Chi-cuadrado", "donde se usa": "H1, H3, H4 (x2), H5 (x2), H7, H8 (x2)"},
])
display(arbol)

print("Contrastes por prueba:  9 chi-cuadrado  +  1 correlacion  +  1 Z test  +  1 ANOVA  =  12")
print()
print("El T Test no se usa porque su condicion es la muestra pequena, y en este dataset")
print(f"el grupo mas pequeno de cualquier comparacion de dos medias tiene {n_min:,} contratos.")
print("Con ese n el arbol manda al Z Test, que es el que se aplica en H2b.")

# --- Paso 2 del procedimiento: los criterios de decision, antes de ver los datos ---
print("\nCriterios de decision fijados a priori en src/config.py:")
criterios = pd.DataFrame([
    {"criterio": "nivel de significancia", "simbolo": "alfa", "valor": config.ALFA,
     "se aplica a": "todas las pruebas primarias"},
    {"criterio": "relevancia practica", "simbolo": "V de Cramer",
     "valor": config.V_MINIMA_UTIL, "se aplica a": "los nueve contrastes chi-cuadrado"},
    {"criterio": "relevancia practica", "simbolo": "diferencia (pp)",
     "valor": config.DIF_MINIMA_PP, "se aplica a": "H3, diferencia de proporciones"},
    {"criterio": "relevancia practica", "simbolo": "r",
     "valor": config.RHO_MINIMO_UTIL, "se aplica a": "H2, correlacion"},
    {"criterio": "relevancia practica", "simbolo": "factor de valor",
     "valor": config.FACTOR_MINIMO_UTIL, "se aplica a": "H2b, Z test sobre log10(valor)"},
    {"criterio": "relevancia practica", "simbolo": "eta cuadrado",
     "valor": config.ETA2_MINIMO_UTIL, "se aplica a": "H6, ANOVA"},
])
display(criterios)
print("Se declaran aqui, en el notebook de estrategia, para que el notebook 03 los consuma")
print("desde config y no pueda ajustarlos en funcion de lo que salga.")

,rama del arbol,prueba,donde se usa
0,numerica vs numerica,Correlation Test,H2 (decil de valor vs tasa)
1,"numerica vs categorica, <=2 grupos, muestras g...",Z Test,"H2b (log10 del valor, con y sin desviacion)"
2,"numerica vs categorica, <=2 grupos, muestras p...",T Test,"no aplica: ningun grupo baja de 26,603 obs."
3,"numerica vs categorica, >2 grupos",ANOVA,H6 (dias adicionados por sector)
4,categorica vs categorica,Chi-cuadrado,"H1, H3, H4 (x2), H5 (x2), H7, H8 (x2)"


Contrastes por prueba:  9 chi-cuadrado  +  1 correlacion  +  1 Z test  +  1 ANOVA  =  12

El T Test no se usa porque su condicion es la muestra pequena, y en este dataset
el grupo mas pequeno de cualquier comparacion de dos medias tiene 26,603 contratos.
Con ese n el arbol manda al Z Test, que es el que se aplica en H2b.

Criterios de decision fijados a priori en src/config.py:


,criterio,simbolo,valor,se aplica a
0,nivel de significancia,alfa,0.05,todas las pruebas primarias
1,relevancia practica,V de Cramer,0.10,los nueve contrastes chi-cuadrado
2,relevancia practica,diferencia (pp),2.00,"H3, diferencia de proporciones"
3,relevancia practica,r,0.30,"H2, correlacion"
4,relevancia practica,factor de valor,1.50,"H2b, Z test sobre log10(valor)"
5,relevancia practica,eta cuadrado,0.06,"H6, ANOVA"


Se declaran aqui, en el notebook de estrategia, para que el notebook 03 los consuma
desde config y no pueda ajustarlos en funcion de lo que salga.


---
## Siguiente paso

El notebook [`03_desarrollo_estrategia.ipynb`](03_desarrollo_estrategia.ipynb) implementa esta
estrategia: contrasta las ocho hipótesis con sus tamaños de efecto, hace el
control por composición y reporta tanto los resultados significativos como los que
no lo son.